# 02b - Dados para os testes de hipóteses

Os testes de hipóteses (Fase 2 do enunciado, *Hypothesis testing*) são feitos em **R**, em `R/testes_hipoteses.Rmd`. O R não lê os pickles do pandas, por isso este notebook grava em CSV só as colunas que cada teste usa.

| Hipótese | Dados | Colunas |
|---|---|---|
| (a) O DTI é mais alto nos empréstimos em *Default* | treino da classificação (36 meses, 2007-06 a 2011-12) | `dti`, `target` |
| (b) O `purpose` está associado ao *default* | treino da classificação | `purpose`, `target` |
| (c) Aceites e rejeitados diferem no montante pedido e no *risk score* | aceites e rejeitados do desenvolvimento (2007-06 a 2014-12) | `amount`, `risk_score`, `date` |

**Regras:**
- Só dados de desenvolvimento. O teste final (2015) não é lido.
- As hipóteses com o desfecho usam só o treino da classificação: é o único conjunto em que o desfecho de cada empréstimo já era observável na data de treino simulada (`desenho_split.md`).
- Nada é alterado: os valores já foram limpos no pré-processamento (`01`, `01b`).

**Antes de correr:** `01_preprocessamento.ipynb` e `01b_rejeitados.ipynb`.

In [1]:
# Setup: imports e caminhos do projeto
import gc
import sys
from pathlib import Path

# Importar o pacote do projeto.
try:
    import lending
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "src" / "lending").is_dir():
            sys.path.insert(0, str(parent / "src"))
            break
    import lending

import pandas as pd

from lending import config
from lending.hypotheses import SAME_SCALE_END, HypothesisDataExporter
from lending.storage import PickleStore

exporter = HypothesisDataExporter()

## 1. Hipóteses sobre o *default*: (a) DTI e (b) `purpose`

Uma linha por empréstimo do treino da classificação. Verifica-se que nenhuma das três colunas tem valores em falta, porque os testes em R não deviam ter de descartar linhas em silêncio.

In [2]:
cls = PickleStore.load(config.CLS_TRAIN_PKL, expected_type=pd.DataFrame)
default_table = exporter.default_table(cls)

assert default_table.notna().all().all(), "há valores em falta nas colunas dos testes"
print(f"{len(default_table):,} empréstimos | categorias de purpose: {default_table['purpose'].nunique()}")
print(default_table[config.CLASS_COL].value_counts().reindex(config.CLASS_ORDER).to_string())

path = exporter.export(default_table, "default")
print(f"Gravado: {path.relative_to(config.PROJECT_ROOT)}")

del cls
_ = gc.collect()

31,522 empréstimos | categorias de purpose: 14
target
Good standing    27646
Default           3876
Gravado: data\processed\hipoteses\default.csv


## 2. Aceites vs rejeitados: (c) montante pedido e *risk score*

As colunas dos dois ficheiros passam a ter o mesmo nome e a mesma unidade:

| Coluna | Aceites | Rejeitados |
|---|---|---|
| `amount` (USD) | `loan_amnt` | `amount_requested` |
| `risk_score` (pontos) | ponto médio de `fico_range_low` e `fico_range_high` | `risk_score` |
| `date` | `issue_d` (mês de financiamento) | `application_date` (dia do pedido) |

**Mudança de escala do *risk score*:** nos rejeitados, o *risk score* é FICO até 2013-11-05 e VantageScore a partir de 2013-11-06 (`rejeitados.md`, secção 5.2). As duas escalas não são comparáveis, por isso a comparação do *risk score* usa só `same_scale_period`: os meses completos antes da mudança (2007-06 a 2013-10), **nos dois grupos**, para comparar o mesmo período de calendário. Os 5 dias de novembro de 2013 ficam de fora porque o `issue_d` dos aceites só tem o mês.

O montante não depende da escala e usa todo o desenvolvimento.

In [3]:
accepted = PickleStore.load(config.ACCEPTED_DEV_PKL, expected_type=pd.DataFrame)
rejected = PickleStore.load(config.REJECTED_DEV_PKL, expected_type=pd.DataFrame)
groups = exporter.accepted_vs_rejected(accepted, rejected)
del accepted, rejected
_ = gc.collect()

# Nos meses usados para o risk score, os rejeitados têm de estar todos na escala FICO
print(f"Período com a mesma escala: até {SAME_SCALE_END - pd.Timedelta(days=1):%Y-%m-%d}")

# Quantas linhas entra em cada comparação (valores em falta ficam de fora de cada teste)
pd.DataFrame({
    "linhas": groups.groupby("group").size(),
    "amount preenchido": groups.groupby("group")["amount"].count(),
    "linhas no período FICO": groups.groupby("group")["same_scale_period"].sum(),
    "risk_score preenchido no período FICO": groups[groups["same_scale_period"]].groupby("group")["risk_score"].count(),
})

Período com a mesma escala: até 2013-10-31


,linhas,amount preenchido,linhas no período FICO,risk_score preenchido no período FICO
group,,,,
aceite,466124,466124,200882,200882
rejeitado,3424228,3424226,1387616,1267436


In [4]:
path = exporter.export(groups, "aceites_rejeitados")
print(f"Gravado: {path.relative_to(config.PROJECT_ROOT)} ({path.stat().st_size / 1e6:,.0f} MB)")

Gravado: data\processed\hipoteses\aceites_rejeitados.csv (157 MB)


## 3. Pipeline e PCA em R

Em `R/pipeline_pca.Rmd`, o mesmo tipo de pipeline do Python (logaritmo -> imputação pela mediana -> padronização) é feito com o pacote `recipes`, ajustado só no treino de cada *fold* da validação temporal, e usado para um PCA.

Usa-se o treino da classificação com as **mesmas colunas numéricas do PCA da EDA** (`02_eda.ipynb`), para se poder comparar o PCA em R com o do Python. `issue_d` serve para construir os *folds* anuais (validação em 2009, 2010 e 2011, como no `01c_pipeline.ipynb`).

In [5]:
# As mesmas colunas numéricas do PCA da classificação no notebook 02 (CLS_NUM)
PIPELINE_COLUMNS = ["loan_amnt", "annual_inc", "dti", "fico_range_low", "revol_util", "revol_bal",
                    "open_acc", "total_acc", "inq_last_6mths", "delinq_2yrs", "pub_rec",
                    "int_rate", "installment", "emp_length"]

cls = PickleStore.load(config.CLS_TRAIN_PKL, expected_type=pd.DataFrame)
pipeline_table = exporter.pipeline_table(cls, PIPELINE_COLUMNS)
del cls
_ = gc.collect()

print(f"{len(pipeline_table):,} empréstimos x {pipeline_table.shape[1]} colunas")
print("Empréstimos por ano (define os folds):",
      pipeline_table["issue_d"].dt.year.value_counts().sort_index().to_dict())
# Valores em falta: ficam para a imputação do pipeline (aprendida no treino de cada fold)
print("Valores em falta:", pipeline_table.isna().sum()[lambda s: s > 0].to_dict())

path = exporter.export(pipeline_table, "pipeline_classificacao")
print(f"Gravado: {path.relative_to(config.PROJECT_ROOT)}")

31,522 empréstimos x 16 colunas
Empréstimos por ano (define os folds): {2007: 603, 2008: 2388, 2009: 5280, 2010: 9153, 2011: 14098}
Valores em falta: {'annual_inc': 4, 'revol_util': 86, 'open_acc': 29, 'total_acc': 29, 'inq_last_6mths': 29, 'delinq_2yrs': 29, 'pub_rec': 29, 'emp_length': 835}
Gravado: data\processed\hipoteses\pipeline_classificacao.csv


## 4. Próximo passo

Os testes estão em `R/testes_hipoteses.Rmd` e o pipeline com PCA em `R/pipeline_pca.Rmd`. Para gerar os relatórios HTML (`reports/testes_hipoteses.html` e `reports/pipeline_pca.html`), a partir da pasta do projeto:

```
Rscript -e "rmarkdown::render('R/testes_hipoteses.Rmd', output_dir = 'reports')"
Rscript -e "rmarkdown::render('R/pipeline_pca.Rmd', output_dir = 'reports')"
```